# Notebook 01: Cohort Selection Flowchart and Baseline Characteristics

**Study:** Machine Learning Prediction of Absence of Formal Healthcare Seeking for Acutely Ill Children in Sub-Saharan Africa: A 38-Country IECV Study  
**Target Journal:** *The Lancet Digital Health*  
**DOI:** [10.5281/zenodo.10892345](https://doi.org/10.5281/zenodo.10892345)

This notebook reproduces:
1. **Figure 1:** Study selection and cohort attrition flowchart (358,499 records -> 118,910 acutely ill children).
2. **Table 1:** Baseline clinical and socioeconomic characteristics stratified by care-seeking status (met vs unmet need) with two-sample hypothesis tests.
3. **Figure 2:** Geographical distribution of national sample sizes and unmet healthcare need prevalence across all 38 nations.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set project paths
REPO_ROOT = Path("..").resolve()
TABLES_DIR = REPO_ROOT / "results" / "tables"
FIGURES_DIR = REPO_ROOT / "results" / "figures"
DATA_DIR = REPO_ROOT / "data"

# Matplotlib typography and styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
print("Environment initialized successfully.")

## 1. Study Selection Flowchart (Figure 1)

Participant derivation flow:
- **Total Birth History Records:** 358,499
- **Excluded Deceased Children:** 19,929 (`B5 != 1`)
- **Surveyed Living Children under 5:** 338,570 (`B5 == 1` and `B19 < 60` / `B8 < 5`)
- **Excluded without Acute Illness:** 219,660 (no fever, diarrhea, or cough in 2-week recall)
- **Primary Analytic Cohort:** 118,910 (35.1% of living children)
- **Care-Seeking Outcome:**
  - **Unmet Need ($Y = 1$):** 70,535 (59.32%, 95% CI: 59.04–59.60%)
  - **Care Sought / Met Need ($Y = 0$):** 48,375 (40.68%)

In [ ]:
# Render Figure 1: Flowchart Diagram
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
ax.axis('off')

boxes = [
    ("All Birth History Records Evaluated across 38 National Surveys\nN = 358,499", 0.5, 0.90, '#e8f0fe', '#1a73e8'),
    ("Surveyed Living Children Under 5 Years of Age\nN = 338,570 (94.4%)", 0.5, 0.68, '#e8f0fe', '#1a73e8'),
    ("Primary Analytic Cohort: Acutely Ill Children (2-week recall)\nN = 118,910 (35.1% of living children under 5)", 0.5, 0.44, '#e6f4ea', '#137333'),
    ("Unmet Healthcare Need (Y = 1)\nNo formal medical advice or care sought\nn = 70,535 (59.32%)", 0.25, 0.18, '#fce8e6', '#c5221f'),
    ("Met Healthcare Need (Y = 0)\nAdvice/care sought from formal medical provider\nn = 48,375 (40.68%)", 0.75, 0.18, '#e6f4ea', '#137333')
]

exclusions = [
    ("Excluded: Deceased children\nn = 19,929 (5.6%)", 0.82, 0.79),
    ("Excluded: No qualifying acute illness\n(No fever, diarrhea, or cough in 2-week recall)\nn = 219,660 (64.9%)", 0.82, 0.56)
]

for text, x, y, bg, border in boxes:
    ax.text(x, y, text, ha='center', va='center', fontsize=10.5, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=bg, edgecolor=border, lw=2))

for text, x, y in exclusions:
    ax.text(x, y, text, ha='center', va='center', fontsize=9.5, style='italic',
            bbox=dict(boxstyle='square,pad=0.5', facecolor='#f1f3f4', edgecolor='#5f6368', lw=1.2, ls='--'))

# Connective arrows
ax.annotate('', xy=(0.5, 0.74), xytext=(0.5, 0.84), arrowprops=dict(arrowstyle="->", lw=2, color='#202124'))
ax.annotate('', xy=(0.66, 0.79), xytext=(0.5, 0.79), arrowprops=dict(arrowstyle="->", lw=1.5, color='#5f6368', ls=':'))
ax.annotate('', xy=(0.5, 0.50), xytext=(0.5, 0.62), arrowprops=dict(arrowstyle="->", lw=2, color='#202124'))
ax.annotate('', xy=(0.66, 0.56), xytext=(0.5, 0.56), arrowprops=dict(arrowstyle="->", lw=1.5, color='#5f6368', ls=':'))
ax.annotate('', xy=(0.25, 0.25), xytext=(0.45, 0.38), arrowprops=dict(arrowstyle="->", lw=2, color='#c5221f'))
ax.annotate('', xy=(0.75, 0.25), xytext=(0.55, 0.38), arrowprops=dict(arrowstyle="->", lw=2, color='#137333'))

plt.title("Figure 1: Study Selection and Cohort Attrition Flowchart", fontsize=13, fontweight='bold', pad=15)
plt.savefig(FIGURES_DIR / "Figure1_flowchart_reproduced.png", dpi=300, bbox_inches='tight')
plt.show()

## 2. Table 1: Baseline Characteristics of Primary Analytic Cohort (N = 118,910)

In [ ]:
table1_path = TABLES_DIR / "Table_1.csv"
if not table1_path.exists():
    table1_path = TABLES_DIR / "table1_baseline_characteristics.csv"

df_t1 = pd.read_csv(table1_path)
display(df_t1)

## 3. Figure 2: Geographic Distribution of Sample Size and Unmet Need Prevalence

In [ ]:
table3_path = TABLES_DIR / "Table_3.csv"
if not table3_path.exists():
    table3_path = TABLES_DIR / "table3_country_iecv_breakdown.csv"

df_iecv = pd.read_csv(table3_path)
df_countries = df_iecv[df_iecv['ISO/DHS Code'] != 'Summary'].sort_values('Sample Size (N)', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 12), dpi=300, sharey=True)

# Panel A: Sample Size
ax1.barh(df_countries['Country'], df_countries['Sample Size (N)'], color='#2b5c8f', edgecolor='#1a365d')
ax1.set_xlabel('Number of Acutely Ill Children (N)', fontsize=11, fontweight='bold')
ax1.set_title('A: National Cohort Sample Sizes', fontsize=12, fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.5)

# Panel B: Prevalence Dot Plot
ax2.scatter(df_countries['Prevalence (%)'], df_countries['Country'], color='#c5221f', s=60, zorder=3)
ax2.axvline(59.32, color='gray', linestyle='--', lw=1.5, label='Continental Mean (59.32%)')
ax2.set_xlabel('Unmet Healthcare Need Prevalence (%)', fontsize=11, fontweight='bold')
ax2.set_title('B: Country-Specific Unmet Need Prevalence', fontsize=12, fontweight='bold')
ax2.set_xlim(20, 85)
ax2.legend(loc='lower right')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Figure 2: Geographical Distribution of Cohort Sample Size and Unmet Need Prevalence', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure2_geographical_distribution_reproduced.png", dpi=300, bbox_inches='tight')
plt.show()